# Utveksling av data med JSON

Det er mange grunner til å utveksle data mellom datasystemer:

* Handelspartnere sender tilbud, ordrer, pakksedler og fakturaer elektronisk. Det er billigere og raskere enn papir, og ingen trenger å taste inn det samme flere ganger.
* En bedrift samler data fra avdelinger og datterselskaper for å lage samlede salgs- og regnskapstall.
* Nettleseren laster ned nettsider fra webservere.
* Apper henter data fra databaser i skyen – for eksempel togtider i Vy-appen.
* Når et gammelt datasystem byttes ut, må dataene flyttes til det nye.

Felles for alle: avsenderen må sende dataene i et **format mottakeren forstår**.

> Kjør cellene i rekkefølge med **Shift + Enter**.

## 1 Fra EDI til JSON

| Tid | Standard | Kjennetegn |
|---|---|---|
| 1970-tallet → | **EDI** (*Electronic Data Interchange*) | avtalte, strukturerte formater mellom kjøpers og selgers systemer. SWIFT, som bankene bruker til utenlandsbetalinger, er et eksempel |
| 1990-tallet | **HTML** og HTTP | nettsider overføres fra tjenere (*servers*) til klienter (*clients*). Tjener og klient kan kjøre helt ulik programvare og maskinvare |
| slutten av 1990-tallet | **XML** | en standard for å lage egne markeringsspråk med egne tagger. HTML er laget for å *vise* informasjon, XML for å *beskrive* (strukturere og lagre) den |
| 2000-tallet | **JSON** (*JavaScript Object Notation*) | enklere enn XML, tekstbasert, støttes av de fleste språk |

En **protokoll** er regler for hvordan sender og mottaker utveksler data. EDI skiller seg fra e-post ved at dokumentene har et avtalt format og behandles automatisk av maskiner.

JSON er blitt populært sammen med JavaScript. Det er enklere enn XML, men ikke bedre til alt.

## 2 JSON-formatet

JSON bygger på to strukturer:

* **navn/verdi-par** – tilsvarer en ordbok (`dict`) i Python
* **ordnede lister** – tilsvarer en liste (`list`)

Regler:
* `[ ]` omslutter lister og `{ }` omslutter objekter.
* Tekst **må** stå i "doble anførselstegn" – 'apostrofer' godtas ikke.
* Verdier kan være objekter, lister, tall, tekst, `true`, `false` eller `null`.
* Elementene skilles med komma (men ikke komma etter det siste!).

Her er en JSON-fil med en liste med fire objekter:

In [ ]:
%%writefile personer.json
[
    {"navn": "Knut",  "alder": 32},
    {"navn": "Tiril", "alder": 28},
    {"navn": "Anne",  "alder": 41},
    {"navn": "Oscar", "alder": 35}
]

## 3 Lese JSON – deserialisering

Vi importerer modulen `json`, åpner filen og bruker `json.load()` til å gjøre teksten om til Python-objekter. Det kalles **deserialisering** eller **parsing**, og følger tabellen:

| JSON | Python |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number (int) | `int` |
| number (real) | `float` |
| true / false | `True` / `False` |
| null | `None` |

In [ ]:
import json

with open('personer.json', encoding='utf-8') as fil:
    data = json.load(fil)

print(data)
print(type(data))
for obj in data:
    print(type(obj))

Vi har fått en liste med fire ordbøker, og kan skrive den ut som i 2.2 Tupler og ordbøker:

In [ ]:
print('Navneliste:\n**********')
for obj in data:
    for nøkkel, verdi in obj.items():
        print(f'{nøkkel.capitalize():6}: {verdi}')
    print()

Vi **må** bruke `json.load()`. Med `fil.read()` fra 1.6 Filer får vi bare en tekst, ikke en liste med ordbøker:

In [ ]:
with open('personer.json', encoding='utf-8') as fil:
    tekst = fil.read()
print(type(tekst))
print(type(json.loads(tekst)))     # loads = load string

## 4 Skrive JSON – serialisering

Den andre veien: Python-objekter lagres som JSON-tekst med `json.dump()`. Det kalles **serialisering**. `indent=2` gir pene innrykk, og `ensure_ascii=False` gjør at æ, ø og å skrives som de er.

In [ ]:
ordbok = {
    'fornavn': 'Per',
    'yrke': 'Snekker',
    'fødselsår': 1992,
    'poststed': 'Ålesund',
    'sertifikat': True,
    'barn': None,
}

with open('per.json', 'w', encoding='utf-8') as fil:
    json.dump(ordbok, fil, indent=2, ensure_ascii=False)

print(open('per.json', encoding='utf-8').read())

Uten `ensure_ascii=False`:

In [ ]:
print(json.dumps({'poststed': 'Ålesund'}))

## 5 Egne klasser og pickle

Ikke alle Python-objekter kan lagres som JSON. Objekter av egne klasser (2.6) gir feil:

In [ ]:
class Hund:
    def __init__(self, navn):
        self.navn = navn

try:
    json.dumps(Hund('Hans'))
except TypeError as feil:
    print('TypeError:', feil)

To løsninger:

1. Gjør objektet om til en ordbok selv, for eksempel med `vars(objekt)` (eller `objekt.__dict__`).
2. Bruk modulen `pickle`, som serialiserer til **binære** filer. Pickle-filer er Python-spesifikke og bør bare åpnes hvis du stoler på kilden – de kan inneholde kode som kjøres.

In [ ]:
print(json.dumps(vars(Hund('Hans'))))

import pickle
with open('hunder.pkl', 'wb') as fil:
    pickle.dump([Hund('Hans'), Hund('Hege')], fil)

with open('hunder.pkl', 'rb') as fil:
    hunder = pickle.load(fil)
print([h.navn for h in hunder])

**Oppsummert:** JSON er et format for å utveksle data mellom datasystemer som kan kjøre ulik programvare på ulik maskinvare. Det handler om hvordan innholdet er *formatert*. Innholdet kan ligge i en fil eller være svaret fra et API-kall. Filer kan overføres med FTP, HTTP, som e-postvedlegg, på minnepinne osv.

Vedlegg *5.5 Avansert JSON* går dypere inn i hierarkier og flere dimensjoner. I neste notebook møter vi mer kompliserte JSON-strukturer, og lar biblioteker gjøre jobben.

## 6 Oppgaver

### Oppgave 1 – Vekt og høyde
Cellen under lager filen `vekt.json`.

a) Skriv ut hvor mye Elise veier (`Elise veier 59 kg.`). Plukk ut ordboken med Elise fra listen med indeks, og `vekt` fra ordboken.
b) Skriv ut alle navnene med vekt og høyde:
```
Kari
  vekt    : 62
  høyde   : 165
...
```

In [ ]:
%%writefile vekt.json
[
  {"navn": "Kari",   "vekt": 62, "høyde": 165},
  {"navn": "Elise",  "vekt": 59, "høyde": 171},
  {"navn": "Gunnar", "vekt": 84, "høyde": 187},
  {"navn": "Helge",  "vekt": 76, "høyde": 178}
]

In [ ]:
# Oppgave 1

### Oppgave 2 – Registrere personer
Lag en tom liste `personer`. Brukeren skriver inn `navn` og `alder`, og en ordbok med navn og alder legges til listen. Slik fortsetter det til brukeren trykker Enter uten å skrive noe. Da lagres listen i `registrerte.json`:
```json
[
  {
    "navn": "Kari",
    "alder": 23
  },
  ...
]
```

In [ ]:
# Oppgave 2

### Oppgave 3 – Biler
Cellen under lager filen `biler.json`.

a) Skriv ut bilmerkene.
b) Skriv ut informasjon om den andre Ford-modellen i listen (blå Mustang).
c) Skriv ut all informasjonen:
```
Toyota
  modell: Corolla
  farge : Grønn
...
```
d) Lag funksjonen `biler_med_farge(farge: str) -> None` som skriver ut alle modellene med en bestemt farge, eller `Ingen funnet`.

In [ ]:
%%writefile biler.json
{
  "Toyota": [
    {"modell": "Corolla", "farge": "Grønn"},
    {"modell": "Land Cruiser", "farge": "Hvit"}
  ],
  "Ford": [
    {"modell": "Mustang", "farge": "Rød"},
    {"modell": "Mustang", "farge": "Blå"},
    {"modell": "Escort", "farge": "Hvit"},
    {"modell": "Explorer", "farge": "Grå"}
  ],
  "Volkswagen": [
    {"modell": "ID.4", "farge": "Grå"},
    {"modell": "Golf", "farge": "Gul"},
    {"modell": "Caravelle", "farge": "Sølv"}
  ],
  "Porsche": [
    {"modell": "911 Carrera", "farge": "Rød"}
  ]
}

In [ ]:
# Oppgave 3

In [ ]:
def biler_med_farge(farge: str) -> None:
    ...

# biler_med_farge('Rød')
# biler_med_farge('Lilla')